# Stage 2 — SST Human Entities, Roles & Target-Window Tracking

Primary Stage-2 runtime:

```text
Stage-1 target window
→ SST 6-class detections
→ cross-class physical-human consolidation
→ RTMW shared cue cache
→ decision-frame-anchored bidirectional tracking
→ track-level role aggregation
→ EntityTrackState + Stage-3 handoff
```

Stage 2 does **not** assign teams, attacking/defending roles, world XYZ, Body3D, or offside. RTMW runs here only once as shared raw evidence/tracking cue; formal 2D pose quality belongs to Stage 3.


## 0. Imports and mode

Use `MODE="production"` with the actual frozen Stage-1 workspace and explicit model paths. `MODE="legacy_validation"` is included only to regression-test the new contract using the real v0.4 JSON bundled in `samples/legacy_validation`.


In [1]:
from pathlib import Path
import json, sys

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "stage2_entities").is_dir():
    raise RuntimeError("Open this notebook with working directory = stage2_sst_rtmw_v1.0")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from stage2_entities import ReplayContext, replay_context_from_stage1_workspace, extract_analysis_window
from stage2_entities.perception import SSTRTMWStage2Backend
from stage2_entities.legacy_adapter import build_manifest_from_legacy_jsons
from stage2_entities.pipeline import finalize_stage2
from stage2_entities.evaluation import raw_cross_class_duplicate_rate

MODE = "legacy_validation"  # production | legacy_validation
OUTPUT_DIR = PROJECT_DIR / "runs" / "stage2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Production paths
STAGE1_ROOT = Path("../stage_1_camera_v12")
VIDEO_OVERRIDE = None
SST_CHECKPOINT = Path("../weights/model.pth")
RTMW_MODEL = Path("../weights/rtmw_l_384x288.onnx")

print("Project:", PROJECT_DIR)
print("Mode:", MODE)


Project: /mnt/data/stage2_sst_rtmw_v1.0
Mode: legacy_validation


## 1. ReplayContext

Production never selects a new temporal window. It adapts the **actual Stage-1 v12 frozen artifacts**, preserving global frame numbers and raw image coordinates. Stage-1 `original_raw` is mapped to the Stage-2 name `RAW_DISTORTED_PIXEL` without a geometric transform.


In [2]:
if MODE == "production":
    context = replay_context_from_stage1_workspace(STAGE1_ROOT, video_override=VIDEO_OVERRIDE)
else:
    context = ReplayContext.from_json(PROJECT_DIR / "samples" / "replay_context_legacy_validation.json")

context.validate()
print(json.dumps(context.to_dict(), indent=2, ensure_ascii=False))


{
  "schema_version": "1.0-test",
  "video_path": "/nonexistent/football_scene.mp4",
  "video_id": "legacy_validation",
  "fps": 25.0,
  "frame_count": 630,
  "image_width": 1280,
  "image_height": 720,
  "selected_frame": 86,
  "window_start": 85,
  "window_end": 87,
  "shot_start": 0,
  "shot_end": 629,
  "coordinate_space": "RAW_DISTORTED_PIXEL",
  "extra": {}
}


## 2. Produce/reuse the Stage-2 perception manifest

**Production order is important:** SST class-local cleanup → cross-class physical-human consolidation → RTMW. This prevents one physical referee/player or player/goalkeeper duplicate from receiving multiple downstream person hypotheses.


In [3]:
if MODE == "production":
    if not SST_CHECKPOINT.is_file():
        raise FileNotFoundError(f"Missing SST checkpoint: {SST_CHECKPOINT}")
    if not RTMW_MODEL.is_file():
        raise FileNotFoundError(f"Missing RTMW model: {RTMW_MODEL}")

    frame_map = extract_analysis_window(context, OUTPUT_DIR / "input_window_frames")
    backend = SSTRTMWStage2Backend(
        sst_checkpoint=SST_CHECKPOINT,
        rtmw_model=RTMW_MODEL,
        sst_device="auto",
        rtmw_device="cpu",
        player_threshold=0.50,
        goalkeeper_threshold=0.50,
        ball_threshold=0.35,
        referee_threshold=0.55,
        staff_threshold=0.60,
        cross_class_iou_threshold=0.85,
        role_ambiguous_margin=0.05,
        keypoint_threshold=1.0,  # RTMW raw SimCC score, NOT probability
    )
    perception_manifest = backend.process_window(frame_map, OUTPUT_DIR / "perception")
else:
    legacy = sorted((PROJECT_DIR / "samples" / "legacy_validation").glob("*_sst_pose.json"))
    perception_manifest = build_manifest_from_legacy_jsons(legacy, OUTPUT_DIR / "legacy_migration")

print("Perception frames:", len(perception_manifest["frames"]))


Perception frames: 3


## 3. Finalize Stage 2

Tracking is anchored at the user-selected frame. Every final track corresponds to one physical human at `t0`; unmatched people that exist only outside `t0` remain orphan diagnostics rather than entering the Stage-3 candidate set.


In [4]:
state = finalize_stage2(
    context=context,
    perception_manifest=perception_manifest,
    output_dir=OUTPUT_DIR,
    render_video=(MODE == "production"),
)
print(json.dumps({
    "status": state.status,
    "tracks": len(state.tracks),
    "candidate_tracks": len(state.stage3_handoff["candidate_track_ids"]),
    "diagnostics": state.diagnostics,
    "artifacts": state.artifacts,
}, indent=2, ensure_ascii=False))


{
  "status": "VALID",
  "tracks": 10,
  "candidate_tracks": 10,
  "diagnostics": {
    "num_tracks": 10,
    "num_candidate_tracks": 10,
    "num_excluded_tracks": 0,
    "num_degraded_candidate_tracks": 0,
    "candidate_tracks_missing_rtmw_at_selected_frame": [],
    "raw_human_detections": 33,
    "consolidated_human_hypotheses": 31,
    "cross_class_duplicate_groups_collapsed": 2,
    "orphan_observations_by_frame": {
      "86": [],
      "85": [
        {
          "physical_human_id": "human_001",
          "role": "player",
          "bbox_xyxy": [
            0.11400000005960464,
            269.76800537109375,
            13.434000015258789,
            329.04400634765625
          ],
          "detector_score": 0.88592
        }
      ],
      "87": []
    }
  },
  "artifacts": {
    "rtmw_track_cache": "/mnt/data/stage2_sst_rtmw_v1.0/runs/stage2/stage2_rtmw_track_cache.json",
    "entity_track_state": "/mnt/data/stage2_sst_rtmw_v1.0/runs/stage2/stage2_entity_tracks.json",


## 4. Inspect Stage-3 handoff

Only final `player`/`goalkeeper` tracks are Stage-3 candidates. Referee/staff remain audit-visible but are excluded. The RTMW cache is re-keyed by persistent track ID and global frame index; its scores remain raw, uncalibrated SimCC maxima.


In [5]:
handoff_path = Path(state.artifacts["stage3_handoff"])
cache_path = Path(state.artifacts["rtmw_track_cache"])
print(handoff_path.read_text(encoding="utf-8"))
cache = json.loads(cache_path.read_text(encoding="utf-8"))
print("RTMW score semantics:", cache["score_semantics"])
print("RTMW cache tracks:", list(cache["tracks"].keys())[:20])


{
  "stage3_input_ready": true,
  "candidate_track_ids": [
    "track_001",
    "track_002",
    "track_003",
    "track_004",
    "track_005",
    "track_006",
    "track_007",
    "track_008",
    "track_009",
    "track_010"
  ],
  "candidate_tracks_missing_rtmw_at_selected_frame": [],
  "entity_track_state_schema": "entity-track-state-1.0",
  "raw_rtmw_track_cache": "/mnt/data/stage2_sst_rtmw_v1.0/runs/stage2/stage2_rtmw_track_cache.json",
  "note": "RTMW cache is raw shared perception evidence, not Stage-3 quality-accepted pose output."
}
RTMW score semantics: RTMW_RAW_SIMCC_MAX_NOT_CALIBRATED_PROBABILITY
RTMW cache tracks: ['track_001', 'track_002', 'track_003', 'track_004', 'track_005', 'track_006', 'track_007', 'track_008', 'track_009', 'track_010']


## 5. Visual QA (production)

Default visual contains only Player/GK candidate boxes and stable track IDs. The debug visual additionally exposes excluded referee/staff. There are no team colours and no offside graphics in Stage 2.


In [6]:
if MODE == "production":
    from IPython.display import Image, Video, display
    display(Image(filename=state.artifacts["selected_frame_visual"]))
    display(Image(filename=state.artifacts["selected_frame_debug_visual"]))
    if "tracking_video" in state.artifacts:
        display(Video(state.artifacts["tracking_video"], embed=False))
else:
    print("Legacy sample has no portable original video path; visual rendering is intentionally skipped.")


Legacy sample has no portable original video path; visual rendering is intentionally skipped.


## 6. Regression diagnostic for the real v0.4 frame 85

This cell demonstrates why the new consolidation stage exists. It measures raw cross-class duplicates before consolidation. The packaged sample is copied from the supplied v0.4 artifact.


In [7]:
legacy85 = json.loads((PROJECT_DIR / "samples" / "legacy_validation" / "clip_000_source_00000085_sst_pose.json").read_text(encoding="utf-8"))
print(json.dumps(raw_cross_class_duplicate_rate(legacy85["detections"]), indent=2))


{
  "raw_human_detections": 13,
  "detections_in_cross_class_duplicate_pairs": 4,
  "CrossClassDuplicateDetectionRate": 0.3076923076923077,
  "pairs": [
    {
      "a": "player_004",
      "b": "goalkeeper_002",
      "iou": 0.9091783417505255
    },
    {
      "a": "player_009",
      "b": "main_referee_001",
      "iou": 0.9156470613368689
    }
  ]
}


## 7. Automated tests

The tests check the actual Player/Referee and Player/Goalkeeper duplicate examples, migration of frames 85–87, Stage-1 adapter, and a mocked production run proving that consolidation happens **before RTMW**.


In [8]:
import subprocess
completed = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Stage-2 tests failed")


.....                                                                    [100%]
5 passed in 4.20s



## 8. Research gate

A structurally `VALID` runtime is not a scientific validation. Before freezing Stage 2, evaluate at minimum CandidateRecall, RefereeLeakageRate, cross-class duplicate rate, HOTA, AssA, IDF1 and TCR@1s on annotated data. See `STAGE2_SPEC.md`.
